In [1]:
# =========================
# STEP 1: Install Libraries
# =========================
!apt-get update -y
!apt-get install -y fenics gmsh
!pip install meshio

# =========================
# STEP 2: Upload STL File
# =========================
from google.colab import files
uploaded = files.upload()   # Upload your STL (e.g., gyroid.stl)

stl_filename = list(uploaded.keys())[0]
print("Uploaded:", stl_filename)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,986 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:1

Saving FG_gyroid_scaffold 3.stl to FG_gyroid_scaffold 3.stl
Uploaded: FG_gyroid_scaffold 3.stl


In [2]:
%pip install gmsh


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 39.6 MB/s eta 0:00:00


In [3]:
# Fix missing OpenGL dependency
!apt-get update -y
!apt-get install -y libglu1-mesa

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libglu1-mesa
0 upgraded, 1 newly installed, 0 

In [ ]:
import gmsh
import numpy as np

gmsh.initialize()
gmsh.open("/content/FG_gyroid_scaffold 3.stl")

# FIXED classifySurfaces (no keywords)
gmsh.model.mesh.classifySurfaces(
    40 * np.pi / 180,
    True,
    True
)

gmsh.model.mesh.createGeometry()

# Get surfaces
surfaces = gmsh.model.getEntities(2)
surface_tags = [s[1] for s in surfaces]

# Create volume
sl = gmsh.model.geo.addSurfaceLoop(surface_tags)
gmsh.model.geo.addVolume([sl])
gmsh.model.geo.synchronize()

# Generate 3D mesh
gmsh.model.mesh.generate(3)

gmsh.write("mesh.msh")
gmsh.finalize()

In [ ]:
%pip install meshio


In [ ]:
import meshio

mesh = meshio.read("mesh.msh")
print(mesh.cells_dict.keys())

In [ ]:
import meshio

mesh = meshio.read("mesh.msh")

meshio.write(
    "mesh.xdmf",
    meshio.Mesh(
        points=mesh.points,
        cells=[("tetra", mesh.cells_dict["tetra"])]
    )
)

In [ ]:
from dolfin import *

mesh = Mesh()
with XDMFFile("mesh.xdmf") as infile:
    infile.read(mesh)

print("Mesh loaded successfully!")

In [ ]:
# Function space
V = VectorFunctionSpace(mesh, "P", 1)

# Boundary condition (bottom fixed)
def bottom(x, on_boundary):
    return near(x[2], 0) and on_boundary

bc = DirichletBC(V, Constant((0, 0, 0)), bottom)

E = 110e9
nu = 0.34

mu = E / (2*(1+nu))
lmbda = E*nu / ((1+nu)*(1-2*nu))

def epsilon(u):
    return sym(grad(u))

def sigma(u):
    return lmbda*tr(epsilon(u))*Identity(3) + 2*mu*epsilon(u)

u = TrialFunction(V)
v = TestFunction(V)

# Apply downward load
f = Constant((0, 0, -1e6))

a = inner(sigma(u), epsilon(v)) * dx
L = dot(f, v) * dx

u_sol = Function(V)
solve(a == L, u_sol, bc)

print("FEM Solution Complete!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plot(u_sol, title="Displacement Field")
plt.show()

stress = project(sigma(u_sol), TensorFunctionSpace(mesh, "P", 1))

plt.figure()
plot(stress, title="Stress Field")
plt.show()